# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riteshy1526/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use a Random Forest model because the content-refresh lane contains a mix of numeric signals such as search volume, CTR, traffic trends, content age, and update recency. A Random Forest can capture non-linear relationships between these signals without requiring a linear relationship between every feature and the target.

The model is used as a ranking and decision-support approach. I will compare it with the Week-4 baseline using the same data, target, split design, and evaluation metric. The goal is not to choose the most complex model, but to check whether the model provides a useful improvement over the baseline.

In [1]:
# Section 1: Imports and dataset loading

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped split by `client_id` so that pages from the same client do not appear in both the training and evaluation sets.

This is a more honest validation design for the content-refresh problem because the model should be tested on clients it did not see during training. This reduces the risk of learning client-specific patterns and gives a better estimate of how the approach may generalize to unseen clients.

The same evaluation split will be used when comparing the model with the Week-4 baseline.

In [2]:
# Section 2: Grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Check the grouping column
print("Number of unique clients:", df["client_id"].nunique())

# 80% training, 20% evaluation
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Training clients:",
    train_df["client_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_id"].nunique()
)

# Verify that no client appears in both sets
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", len(overlap))

assert len(overlap) == 0, "Data leakage: client overlap detected!"

print("Grouped split check passed.")

Number of unique clients: 32
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Grouped split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I train a Random Forest model using features that are available before a content-refresh decision is made. I exclude identifiers and fields that directly describe the target or could introduce leakage.

The model is evaluated on the same grouped test split used for validation. I compare its NDCG with the Week-4 baseline using the same test rows.

NDCG is used because this lane is a ranking problem: the goal is to place higher-priority content opportunities near the top of the review queue.

In [3]:
# Section 3: Prepare data and train the Random Forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import ndcg_score

target_col = "trend_pct"

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_cols = [
    col for col in feature_cols
    if col in df.columns
]

# Remove rows with missing target.
# We do NOT impute the target.
train_valid = train_df.dropna(subset=[target_col]).copy()
test_valid = test_df.dropna(subset=[target_col]).copy()

X_train = train_valid[feature_cols]
y_train = train_valid[target_col]

X_test = test_valid[feature_cols]
y_test = test_valid[target_col]

print("Training rows after target check:", len(train_valid))
print("Test rows after target check:", len(test_valid))

print("Missing training targets:", y_train.isna().sum())
print("Missing test targets:", y_test.isna().sum())

# Impute missing feature values using training medians only.
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Number of features:", len(feature_cols))
print("Training shape:", X_train_imp.shape)
print("Test shape:", X_test_imp.shape)

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

model.fit(X_train_imp, y_train)

# Generate predictions
model_pred = model.predict(X_test_imp)

print("\nModel training complete.")
print("Predictions generated:", len(model_pred))

Training rows after target check: 21133
Test rows after target check: 5479
Missing training targets: 0
Missing test targets: 0
Number of features: 28
Training shape: (21133, 28)
Test shape: (5479, 28)

Model training complete.
Predictions generated: 5479


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

target_col = "trend_pct"

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_cols = [col for col in feature_cols if col in df.columns]

# Remove rows with missing target
train_valid = train_df.dropna(subset=[target_col]).copy()
test_valid = test_df.dropna(subset=[target_col]).copy()

X_train = train_valid[feature_cols]
y_train = train_valid[target_col]

X_test = test_valid[feature_cols]
y_test = test_valid[target_col]

# Impute missing feature values using training data only
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

model.fit(X_train_imp, y_train)

print("Model training complete.")
print("Number of features:", len(feature_cols))

Model training complete.
Number of features: 28


In [5]:
# ---------------------------------------------------------
# NDCG comparison
# ---------------------------------------------------------

# Convert relevance to a non-negative ranking scale.
# Adding a constant preserves the ordering of all observations.
relevance = -eval_df["trend_pct"]

relevance = (
    relevance - relevance.min()
)

# Safety checks
print("Minimum relevance:", relevance.min())
print("Maximum relevance:", relevance.max())

assert relevance.min() >= 0
assert baseline_score.notna().all()
assert model_score.notna().all()

# Calculate NDCG
baseline_ndcg = ndcg_score(
    [relevance.to_numpy()],
    [baseline_score.to_numpy()]
)

model_ndcg = ndcg_score(
    [relevance.to_numpy()],
    [model_score.to_numpy()]
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "NDCG": [
        baseline_ndcg,
        model_ndcg
    ]
})

comparison["NDCG"] = comparison["NDCG"].round(4)

display(comparison)

difference = model_ndcg - baseline_ndcg

print("NDCG difference:", round(difference, 4))

if difference > 0:
    print(
        "Observed result: Random Forest ranked the measured "
        "opportunities better than the baseline on this split."
    )
elif difference < 0:
    print(
        "Observed result: Random Forest ranked the measured "
        "opportunities below the baseline on this split."
    )
else:
    print(
        "Observed result: Random Forest and baseline had "
        "the same NDCG on this split."
    )

NameError: name 'eval_df' is not defined

### Result interpretation

The comparison uses the same eligible test rows and the same NDCG metric for both approaches.

The NDCG difference is interpreted as an observed ranking difference on this evaluation split. A higher score indicates better alignment with the measured ranking target on this test set, but it does not establish a causal improvement from refreshing content.

If the Random Forest does not outperform the baseline, the baseline remains a useful and simpler decision rule. Model complexity is not treated as an improvement by itself.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The error analysis checks where the Random Forest ranking differs from the observed ranking signal and which features the model relies on most.

I focus on the size and direction of the ranking differences rather than only reporting the overall NDCG. Feature importance is used as an interpretation aid, not as evidence of causality.

The results should be read as directional evidence from this evaluation split. A feature being important to the model does not mean that changing that feature will cause better performance.

In [ ]:
# Section 4: Error analysis and feature interpretation

# Create an analysis table for the eligible evaluation rows

error_analysis = eval_df[
    [
        "content_id",
        "trend_pct",
        "days_since_last_update",
        "content_age_days",
        "ctr",
        "search_volume"
    ]
].copy()

error_analysis["actual_relevance"] = relevance.to_numpy()
error_analysis["model_score"] = model_score.to_numpy()
error_analysis["baseline_score"] = baseline_score.to_numpy()

# Rank both approaches
error_analysis["actual_rank"] = (
    error_analysis["actual_relevance"]
    .rank(ascending=False, method="average")
)

error_analysis["model_rank"] = (
    error_analysis["model_score"]
    .rank(ascending=False, method="average")
)

error_analysis["baseline_rank"] = (
    error_analysis["baseline_score"]
    .rank(ascending=False, method="average")
)

# Absolute ranking error
error_analysis["model_rank_error"] = (
    error_analysis["model_rank"]
    - error_analysis["actual_rank"]
).abs()

error_analysis["baseline_rank_error"] = (
    error_analysis["baseline_rank"]
    - error_analysis["actual_rank"]
).abs()

# Largest model ranking errors
largest_errors = (
    error_analysis
    .sort_values("model_rank_error", ascending=False)
    .head(10)
)

print("Top 10 largest model ranking differences:")

display(
    largest_errors[
        [
            "content_id",
            "actual_rank",
            "model_rank",
            "baseline_rank",
            "model_rank_error",
            "trend_pct",
            "days_since_last_update",
            "content_age_days",
            "ctr"
        ]
    ]
)

Top 10 largest model ranking differences:


,content_id,actual_rank,model_rank,baseline_rank,model_rank_error,trend_pct,days_since_last_update,content_age_days,ctr
13938,content_6eb16d7a88ae,206.0,5302.0,274.5,5096.0,-100.0,22,460,0.0
10771,content_70e80d66f0b0,206.0,5256.0,3182.5,5050.0,-100.0,8,145,0.0
26543,content_7764f406228e,206.0,5186.0,290.5,4980.0,-100.0,104,127,0.0
15215,content_b774a3d92c71,460.0,5239.0,2065.0,4779.0,-93.8,20,90,0.0
17919,content_6a552ab4e642,416.0,5194.0,2040.0,4778.0,-98.4,20,90,0.0
23704,content_84bc4706ba53,546.5,5270.0,2580.0,4723.5,-88.1,20,90,0.0
10727,content_5723e05acdc6,433.0,4997.0,2048.0,4564.0,-96.7,20,90,0.0
10853,content_d62bcceebda5,541.0,5102.0,2137.0,4561.0,-88.5,20,90,0.0
23286,content_fbad54d7a5dc,534.0,5068.0,2134.0,4534.0,-88.9,20,90,0.0
21563,content_09f7b75a09a5,466.5,4987.0,1803.0,4520.5,-93.5,20,91,0.0


In [ ]:
# Feature importance from the trained Random Forest

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top model features:")

display(
    feature_importance.head(10)
)

Top model features:


,feature,importance
0,impressions_last_30d,0.570029
1,avg_position,0.122733
2,impressions_90d,0.073263
3,impressions_prev_30d,0.051688
4,scroll_events_90d,0.025829
5,content_age_days,0.015888
6,days_with_impressions,0.015629
7,char_count,0.015439
8,pageviews_90d,0.015393
9,word_count,0.014812


### Interpretation

The error analysis shows that the model does not rank every page in the same order as the measured relevance signal. The largest ranking differences identify cases where the model should be treated cautiously and reviewed by a human.

The feature-importance results show which available signals the Random Forest used most strongly for its predictions. These are associations within the model and should not be interpreted as causal effects.

The model is therefore useful only as a prioritization aid. The observed NDCG result should be considered together with the baseline comparison and error analysis rather than used as a standalone claim.

## Self - check

- [ ] Every section contains both Markdown reasoning and executable code.
- [ ] The notebook uses a grouped client-level evaluation split.
- [ ] Missing target values were excluded rather than artificially imputed.
- [ ] The Random Forest was compared with the baseline using the same eligible evaluation rows.
- [ ] NDCG was used consistently for the ranking comparison.
- [ ] Errors and feature importance were inspected.
- [ ] Claims use careful language such as observed, measured, directional, and decision-support.
- [ ] No client names, private queries, or unnecessary URLs are included.
- [ ] The notebook runs from top to bottom without errors.

In [ ]:
# Final W05 self-check

checks = {
    "data_loaded": len(df) > 0,
    "grouped_split_created": (
        len(set(train_df["client_id"]) &
            set(test_df["client_id"])) == 0
    ),
    "model_trained": hasattr(model, "feature_importances_"),
    "predictions_created": len(model_pred) == len(test_valid),
    "comparison_created": "comparison" in globals(),
    "feature_importance_created": "feature_importance" in globals(),
    "error_analysis_created": "error_analysis" in globals(),
}

self_check = pd.DataFrame(
    checks.items(),
    columns=["check", "passed"]
)

display(self_check)

if self_check["passed"].all():
    print("W05 SELF-CHECK PASSED")
else:
    print("W05 SELF-CHECK FAILED — review the failed checks.")

,check,passed
0,data_loaded,True
1,grouped_split_created,True
2,model_trained,True
3,predictions_created,True
4,comparison_created,True
5,feature_importance_created,True
6,error_analysis_created,True


W05 SELF-CHECK PASSED
